# Session 2 — Data Download and Verification
Run after executing `python src/data/download.py` from the project root.

All data lives under `data/GDSC2/`. The universal cell line key is `cellosaurus_id`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT  = Path(".").resolve().parent
GDSC2 = ROOT / "data" / "GDSC2"
META  = ROOT / "data" / "meta"
print("Root:", ROOT)
print("GDSC2 files:", [f.name for f in GDSC2.iterdir() if f.is_file()])

Root: /home/shoko/multiomics-drug-response
GDSC2 files: ['gene_expression.csv', 'drug_smiles.csv', 'mutations.csv', 'cell_line_names.csv', 'methylation.csv', 'GDSC2.csv', 'copy_number_variation_gistic.csv', 'pathway_features.csv', 'proteomics.csv', 'drug_smilesvec.csv', 'drug_names.csv', 'drug_chemberta_embeddings.csv']


## 1. Cell Line Names (cellosaurus_id mapping)

In [2]:
cl_names = pd.read_csv(GDSC2 / "cell_line_names.csv")
print("Shape:", cl_names.shape)          # expect (969, 3)
print("Columns:", cl_names.columns.tolist())  # cellosaurus_id, cell_line_name, tissue
cl_names.head(5)

Shape: (969, 3)
Columns: ['cellosaurus_id', 'cell_line_name', 'tissue']


,cellosaurus_id,cell_line_name,tissue
0,CVCL_0001,HEL,Blood
1,CVCL_0002,HL-60,Blood
2,CVCL_0004,K-562,Blood
3,CVCL_0008,Daudi,Lymph
4,CVCL_0009,HDLM-2,Lymph


## 2. GDSC2 Drug Response

In [3]:
gdsc2 = pd.read_csv(GDSC2 / "GDSC2.csv", dtype={"pubchem_id": str}, low_memory=False)
print("Shape:", gdsc2.shape)
print("Cell lines:", gdsc2["cell_line_name"].nunique())
print("Drugs:", gdsc2["pubchem_id"].nunique())
print("Columns:", gdsc2.columns.tolist())
gdsc2.head(3)

Shape: (234436, 33)
Cell lines: 969
Drugs: 287
Columns: ['cellosaurus_id', 'cell_line_name', 'pubchem_id', 'drug_name', 'Name', 'SignalQuality', 'pEC50_curvecurator', 'Slope', 'Front', 'Back', 'FoldChange', 'AUC_curvecurator', 'RMSE', 'R2', 'pEC50Error', 'fValue', 'pValue', 'negLog10pValue', 'fValueSAMCorrected', 'RelevanceScore', 'Regulation', 'EC50_curvecurator', 'IC50_curvecurator', 'CELL_LINE_NAME', 'sample', 'drug', 'LN_IC50', 'AUC', 'IC50', 'min_dose_M', 'max_dose_M', 'LN_IC50_curvecurator', 'tissue']


,cellosaurus_id,cell_line_name,pubchem_id,drug_name,Name,SignalQuality,pEC50_curvecurator,Slope,Front,Back,...,CELL_LINE_NAME,sample,drug,LN_IC50,AUC,IC50,min_dose_M,max_dose_M,LN_IC50_curvecurator,tissue
0,CVCL_0001,HEL,5352062,Romidepsin,CVCL_0001|5352062,0.0,8.089522,10.000000,1.002152,0.000100,...,HEL,CVCL_0001,5352062,-5.027970,0.892340,6.552098e-09,1.000530e-11,1.000000e-08,-4.810853,Blood
1,CVCL_0002,HL-60,36314,Paclitaxel,CVCL_0002|36314,0.0,8.396003,10.000000,1.273513,0.081264,...,HL-60,CVCL_0002,36314,-4.906881,0.875756,7.395519e-09,1.000530e-11,1.000000e-08,-5.455631,Blood
2,CVCL_0002,HL-60,5352062,Romidepsin,CVCL_0002|5352062,0.0,8.691853,1.212055,1.027599,0.000100,...,HL-60,CVCL_0002,5352062,-6.094023,0.755228,2.256313e-09,1.000530e-11,1.000000e-08,-6.153727,Blood


## 3. Gene Expression (CCLE RNA-seq)

In [4]:
ge = pd.read_csv(GDSC2 / "gene_expression.csv", index_col=0)
print("Shape:", ge.shape)                        # expect (~1010, ~17000)
print("Index format:", ge.index[:3].tolist())    # expect CVCL_XXXX
ge.iloc[:3, :5]

Shape: (1014, 17738)
Index format: ['CVCL_1104', 'CVCL_1174', 'CVCL_1110']


,cell_line_name,TSPAN6,TNMD,DPM1,SCYL3
cellosaurus_id,,,,,
CVCL_1104,CAL-120,7.632023,2.964585,10.379553,3.614794
CVCL_1174,DMS 114,7.548671,2.777716,11.807341,4.066887
CVCL_1110,CAL-51,8.712338,2.643508,9.880733,3.956230


## 4. Proteomics (ProCan, converted)

In [5]:
prot = pd.read_csv(GDSC2 / "proteomics.csv", index_col=0)
print("Shape:", prot.shape)                       # expect (~860, 8498)
print("Index format:", prot.index[:3].tolist())   # expect CVCL_XXXX
print("Missing values (%):", round(prot.isna().mean().mean() * 100, 2))
prot.iloc[:3, :5]

Shape: (862, 8498)
Index format: ['CVCL_1762', 'CVCL_4384', 'CVCL_X508']
Missing values (%): 38.78


,Q9Y651;SOX21_HUMAN,P37108;SRP14_HUMAN,Q96JP5;ZFP91_HUMAN,Q9Y4H2;IRS2_HUMAN,P36578;RL4_HUMAN
cellosaurus_id,,,,,
CVCL_1762,NaN,6.828022,4.143455,2.215779,7.628785
CVCL_4384,NaN,7.014256,3.858033,2.278077,8.124585
CVCL_X508,NaN,5.285911,3.516953,NaN,7.972680


## 5. Drug Features

In [6]:
smiles = pd.read_csv(GDSC2 / "drug_smiles.csv")
print("Drug SMILES shape:", smiles.shape)
print("Columns:", smiles.columns.tolist())
smiles.head(3)

Drug SMILES shape: (246, 5)
Columns: ['pubchem_id', 'drug_name', 'canonical_smiles', 'cactvs_fingerprint', 'fingerprint']


,pubchem_id,drug_name,canonical_smiles,cactvs_fingerprint,fingerprint
0,5352062,Romidepsin,CC=C1C(=O)NC(C(=O)OC2CC(=O)NC(C(=O)NC(CSSCCC=C...,1111000001111011101110000000000001100000000000...,00000371F07BB800600000000000000000000000000000...
1,36314,Paclitaxel,CC1=C2C(C(=O)C3(C(CC4C(C3C(C(C2(C)C)(CC1OC(=O)...,1111000001111110001111000000000000000000000000...,00000371F07E3C00000000000000000000000000480000...
2,457193,Dactinomycin,CC1C(C(=O)NC(C(=O)N2CCCC2C(=O)N(CC(=O)N(C(C(=O...,1111000001111111111111100000000000000000000000...,00000371F07FFE00000000000000000000000000000162...


## 6. Three-way Overlap (key check)

In [7]:
# Map GDSC2 cell line names to cellosaurus_id
name_to_cvcl = dict(zip(cl_names["cell_line_name"], cl_names["cellosaurus_id"]))
gdsc2_cvcl   = set(gdsc2["cell_line_name"].map(name_to_cvcl).dropna())
ge_cvcl      = set(ge.index.astype(str))
prot_cvcl    = set(prot.index.astype(str))

overlap = gdsc2_cvcl & ge_cvcl & prot_cvcl

print(f"GDSC2 cell lines (cvcl):        {len(gdsc2_cvcl)}")
print(f"Gene expression cell lines:     {len(ge_cvcl)}")
print(f"Proteomics cell lines:          {len(prot_cvcl)}")
print(f"Three-way overlap:              {len(overlap)}  ← this is our working dataset")
print(f"")
print(f"GDSC2 drugs:                    {gdsc2['pubchem_id'].nunique()}")
print(f"Gene expression genes:          {ge.shape[1]}")
print(f"Proteins:                       {prot.shape[1]}")
print(f"Drug-cell line pairs in overlap: {len(gdsc2[gdsc2['cell_line_name'].map(name_to_cvcl).isin(overlap)]):,}")

GDSC2 cell lines (cvcl):        969
Gene expression cell lines:     1010
Proteomics cell lines:          860
Three-way overlap:              836  ← this is our working dataset

GDSC2 drugs:                    287
Gene expression genes:          17738
Proteins:                       8498
Drug-cell line pairs in overlap: 204,931


## 7. Tissue Distribution

In [8]:
# Cell lines in overlap by tissue type
overlap_cl = cl_names[cl_names["cellosaurus_id"].isin(overlap)]
print("Tissue distribution in three-way overlap:")
print(overlap_cl["tissue"].value_counts().to_string())

Tissue distribution in three-way overlap:
tissue
Lung               174
Blood               87
Lymph               52
Brain               49
Breast              48
Skin                45
Colon               44
Ovary               39
Bone                36
Head And Neck       34
Esophagus           34
Stomach             24
Pancreas            23
Kidney              22
Nervous System      18
Bladder             18
Liver               17
Cervix              16
Uterus              15
Thyroid             12
Soft Tissue         10
Prostate             8
Muscle               7
Embryonic            2
Adrenal Gland        1
Small Intestine      1
